# Tuned Lens per Layer — Qwen3.5 9B

Per-layer tuned-lens pipeline from `experiments/tuned_lens_per_layer/` adapted for **Qwen3.5-9B**.
Model loading follows `experiments/notebooks/r2_qwen35_9b.ipynb` (HuggingFace path, `WrappedHFModel`).

Steps:
1. Load Qwen3.5-9B via HuggingFace + wrap as `WrappedHFModel`
2. Generate HMM sequences + get concept token IDs
3. KV-cached chunked forward pass (activations at concept positions only)
4. Train lens variants: `tuned_full`, `tuned_concept`, `tuned_hmm` + `logit` baseline
5. Evaluate all lenses per layer (KL-HMM, KL-final, NLL, top-1)
6. Belief-state R² probe (GPU OLS via `torch.linalg.pinv`)
7. Plots + artifact serialization

In [1]:
%matplotlib inline
%load_ext autoreload
%autoreload 2
import os
# HMM generation uses JAX but is tiny; keep JAX off the GPU so it does not
# fight the Qwen3.5-9B torch model for VRAM (avoids CUDA_OUT_OF_MEMORY from
# JAX's default ~75% preallocation). Must be set before `import jax`.
os.environ['JAX_PLATFORMS'] = 'cpu'
import sys, gc, json, time, copy
from pathlib import Path
import subprocess
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# Locate project root via git (works regardless of Jupyter launch directory)
try:
    PROJECT_ROOT = Path(subprocess.check_output(
        ['git', 'rev-parse', '--show-toplevel'], text=True).strip())
except Exception:
    PROJECT_ROOT = Path.cwd().parent.parent  # fallback: notebook at experiments/notebooks/
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
sys.path.insert(0, str(PROJECT_ROOT))
print(f'Project root: {PROJECT_ROOT}')

Project root: /workspace/geometric-interpretability-LLMs


## Configuration

Edit the variables below to configure the experiment.

In [2]:
# ── HMM process ───────────────────────────────────────────────────────────────
PROCESS_NAME     = 'mess3'
PROCESS_PARAMS   = {'x': 0.05, 'a': 0.85}
VOCAB_TOKENS     = ['F', 'Q', 'V']
SEQ_LENGTH       = 2000
N_SEQUENCES      = 10
N_TRAIN          = 8          # sequences used to train lenses; rest are held-out test
RANDOM_SEED      = 42
TRAIN_POS_WINDOW = None       # [start, end) window for training positions, or None for all

# ── tuned-lens training ───────────────────────────────────────────────────────
TL_EPOCHS     = 50
TL_LR         = 1e-5          # see scripts/diag_last_layer_lr_clip.py: 1e-3 overshoots identity on Qwen3.5-9B (max|x|~228); 1e-5 lands KL~2e-7
TL_BATCH      = 512
TL_OPTIMIZER  = 'adam'        # 'adam' or 'muon'
USE_BF16      = False         # True on A100/H100 for ~16x faster matmuls
CHUNK_SIZE    = 2048          # KV-cached forward-pass chunk size

# ── which lens variants to train ─────────────────────────────────────────────
TRAIN_FULL    = True          # canonical full-vocab KL target (arXiv:2303.08112)
TRAIN_CONCEPT = True          # concept-only logit target (model's final layer)
TRAIN_HMM     = True          # concept-only probability target (HMM oracle)

# ── output ────────────────────────────────────────────────────────────────────
OUTPUT_DIR  = PROJECT_ROOT / 'outputs' / f'qwen35_9b_notebook_{PROCESS_NAME}'
FIGURES_DIR = OUTPUT_DIR / 'figures'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)
print(f'Output: {OUTPUT_DIR}')

Output: /workspace/geometric-interpretability-LLMs/outputs/qwen35_9b_notebook_mess3


## Load Qwen3.5-9B

Loading pattern from `experiments/notebooks/r2_qwen35_9b.ipynb`.
The HuggingFace model is wrapped in `WrappedHFModel` to expose the TransformerLens-compatible
API expected by the tuned-lens training and evaluation functions.

In [3]:
os.environ['HF_HOME'] = '/workspace/hf_cache'   # change to your HF cache path
# os.environ['CUDA_VISIBLE_DEVICES'] = '0'       # uncomment to restrict to one GPU

from transformers import AutoModelForCausalLM, AutoTokenizer
from experiment_utils import WrappedHFModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

MODEL_NAME = 'Qwen/Qwen3.5-9B'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
hf_model  = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,
    attn_implementation='sdpa',
    device_map='auto',
)
hf_model.eval()

N_LAYERS      = len(hf_model.model.layers)
LAYER_INDICES = list(range(N_LAYERS))
print(f'Model: {MODEL_NAME}')
print(f'Layers: {N_LAYERS}, hidden_size: {hf_model.config.hidden_size}')

# Wrap for tuned-lens pipeline compatibility (exposes .unembed.W_U, .ln_final, .cfg)
model = WrappedHFModel(hf_model, tokenizer, device)
print(f'WrappedHFModel: d_model={model.cfg.d_model}, n_layers={model.cfg.n_layers}')

Device: cuda


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/427 [00:00<?, ?it/s]

Model: Qwen/Qwen3.5-9B
Layers: 32, hidden_size: 4096
WrappedHFModel: d_model=4096, n_layers=32


## HMM Data Generation & Token IDs

In [4]:
from data_generation import generate_hmm_sequences
from experiment_utils import get_concept_token_ids

print(f'Generating {N_SEQUENCES} seqs x {SEQ_LENGTH} tokens  [{PROCESS_NAME} {PROCESS_PARAMS}]')
hmm_data = generate_hmm_sequences(
    process_name=PROCESS_NAME,
    process_params=PROCESS_PARAMS,
    n_sequences=N_SEQUENCES,
    seq_length=SEQ_LENGTH,
    random_seed=RANDOM_SEED,
)
tokens_np     = hmm_data.tokens          # (N, L) int
belief_states = hmm_data.belief_states   # (N, L, n_states)
obs_probs     = hmm_data.obs_probs       # (N, L, vocab_size)

walk_concepts = [[VOCAB_TOKENS[int(t)] for t in seq] for seq in tokens_np]
print(f'tokens={tokens_np.shape}, beliefs={belief_states.shape}, obs_probs={obs_probs.shape}')

concept_to_id = get_concept_token_ids(model, VOCAB_TOKENS)
concept_ids   = [concept_to_id[c] for c in VOCAB_TOKENS]
print(f'Concept token IDs: {dict(zip(VOCAB_TOKENS, concept_ids))}')

# Verify each concept maps to a single space-prefixed token
for name, tid in zip(VOCAB_TOKENS, concept_ids):
    decoded = tokenizer.decode([tid])
    print(f'  {name} -> id={tid} -> decoded={decoded!r}')

Generating 10 seqs x 2000 tokens  [mess3 {'x': 0.05, 'a': 0.85}]
tokens=(10, 2000), beliefs=(10, 2000, 3), obs_probs=(10, 2000, 3)
Concept token IDs: {'F': 426, 'Q': 1167, 'V': 629}
  F -> id=426 -> decoded=' F'
  Q -> id=1167 -> decoded=' Q'
  V -> id=629 -> decoded=' V'


## Extract Eval Weights

Extract the concept-column slice of the unembedding matrix and a copy of `ln_final` **while the
model is still on GPU**. These tiny tensors (~KB) replace the backbone (~18 GB) for all evaluation
steps, so the backbone can be freed to CPU after training completes.

In [5]:
from experiments.tuned_lens_per_layer.tuned_lens import (
    EvalWeights, extract_eval_weights,
    TunedLensTranslator,
    train_tuned_lens, train_tuned_lens_concept,
    save_translators, load_translator,
    apply_logit_lens, apply_tuned_lens,
)
from experiments.tuned_lens_per_layer.evaluation import compute_layer_metrics
from experiments.tuned_lens_per_layer import plotting

eval_weights = extract_eval_weights(model, concept_ids)
print(f'Eval weights: W_c{list(eval_weights.W_c.shape)}, device={eval_weights.device}')

Eval weights: W_c[4096, 3], device=cuda:0


## Forward Pass (KV-cached, HuggingFace path)

Registers `register_forward_hook` on each `hf_model.model.layers[l]` to capture
post-layer hidden states only at concept-token positions. Uses native HF `past_key_values`
caching across chunks to avoid materializing the full sequence at once.

This is the same logic as `_forward_pass_hf` in `experiments/tuned_lens_per_layer/pipeline.py`.

In [6]:
from transformers.cache_utils import DynamicCache

def forward_pass_hf(model, walk_concepts, vocab_tokens, layer_indices,
                    n_sequences, chunk_size, concept_ids):
    """KV-cached chunked HF forward pass — uses explicit DynamicCache to avoid
    the hang that occurs with past_key_values=None on Qwen3.5's hybrid arch."""
    hf_m = model._hf_model
    tok  = model._tokenizer
    dev  = model._device

    _nospace = {tok.encode(c, add_special_tokens=False)[-1] for c in vocab_tokens}
    id_set   = set(concept_ids) | _nospace

    seq_acts  = {l: [] for l in layer_indices}
    n_per_seq = []

    for si in tqdm(range(n_sequences), desc='Forward pass (KV-cached)'):
        prompt    = ' ' + ' '.join(walk_concepts[si])
        ids_list  = tok.encode(prompt, add_special_tokens=False)
        input_ids = torch.tensor([ids_list], dtype=torch.long, device=dev)

        ids_np    = np.array(ids_list, dtype=np.int64)
        positions = np.where(np.isin(ids_np, list(id_set)))[0].astype(np.int64)
        n_use     = min(len(positions), len(walk_concepts[si]))
        positions = positions[:n_use]
        n_per_seq.append(int(n_use))

        seq_len    = input_ids.shape[1]
        past_kv    = DynamicCache(config=hf_m.config)   # config-aware cache builds the
                                                # linear-attn cache layers Qwen3.5 needs
        chunk_acts = {l: [] for l in layer_indices}

        for cs in range(0, seq_len, chunk_size):
            ce          = min(cs + chunk_size, seq_len)
            chunk_tok   = input_ids[:, cs:ce]
            local_idx   = np.where(np.isin(np.arange(cs, ce), positions))[0]
            need_cap    = local_idx.size > 0

            captured = {}
            hooks    = []
            if need_cap:
                idx_t = torch.from_numpy(local_idx).to(dev, dtype=torch.long)
                for l in layer_indices:
                    def make_hook(li, _idx=idx_t):
                        def fn(module, inp, out):
                            h = out[0] if isinstance(out, tuple) else out
                            if h is not None and h.dim() == 3:
                                captured[li] = h[0].index_select(0, _idx).detach()
                        return fn
                    hooks.append(hf_m.model.layers[l].register_forward_hook(make_hook(l)))

            with torch.no_grad():
                out = hf_m.model(chunk_tok, past_key_values=past_kv, use_cache=True)
            for h in hooks:
                h.remove()

            if need_cap:
                for l in layer_indices:
                    if l in captured:
                        chunk_acts[l].append(captured[l])
            del captured
            past_kv = out.past_key_values
            del out
            torch.cuda.empty_cache()

        del past_kv

        d = model.cfg.d_model
        for l in layer_indices:
            if chunk_acts[l]:
                merged = torch.cat(chunk_acts[l], dim=0)
                seq_acts[l].append(merged.float().cpu().numpy())
            else:
                seq_acts[l].append(np.zeros((0, d), dtype=np.float32))
            chunk_acts[l].clear()

        del input_ids
        torch.cuda.empty_cache()

    return seq_acts, n_per_seq


In [7]:
t0 = time.time()
seq_activations, n_concepts_list = forward_pass_hf(
    model, walk_concepts, VOCAB_TOKENS,
    LAYER_INDICES, N_SEQUENCES, CHUNK_SIZE, concept_ids,
)
print(f'Forward pass: {time.time()-t0:.1f}s')

seq_len_actual = n_concepts_list[0]
assert all(n == seq_len_actual for n in n_concepts_list), \
    f'Inconsistent concept counts: {set(n_concepts_list)}'
print(f'Concept positions per sequence: {seq_len_actual}')

Forward pass (KV-cached):   0%|          | 0/10 [00:00<?, ?it/s]

Forward pass: 9.8s
Concept positions per sequence: 2000


## Train / Test Split

In [8]:
N_TEST = N_SEQUENCES - N_TRAIN

def concat_seqs(seq_list, start, end, window=None):
    sliced = seq_list[start:end]
    if window is not None:
        ws, we = window
        sliced = [s[ws:we] for s in sliced]
    return np.concatenate(sliced, axis=0)

train_acts = {}
test_acts  = {}
for layer in LAYER_INDICES:
    arrs = seq_activations[layer]
    train_acts[layer] = concat_seqs(arrs, 0, N_TRAIN, window=TRAIN_POS_WINDOW)
    test_acts[layer]  = concat_seqs(arrs, N_TRAIN, N_SEQUENCES)
del seq_activations

final_layer_idx   = LAYER_INDICES[-1]
train_final_resid = train_acts[final_layer_idx]
test_final_resid  = test_acts[final_layer_idx]

# HMM reference probs and next tokens for the test set
test_obs_flat = obs_probs[N_TRAIN:, :seq_len_actual, :].reshape(-1, obs_probs.shape[-1])
test_tokens   = tokens_np[N_TRAIN:, :seq_len_actual]
test_next_full = np.zeros((N_TEST, seq_len_actual), dtype=np.int64)
test_next_full[:, :-1] = test_tokens[:, 1:]
test_next_flat = test_next_full.reshape(-1)

print(f'Train positions: {train_acts[final_layer_idx].shape[0]}')
print(f'Test  positions: {test_acts[final_layer_idx].shape[0]}')

Train positions: 16000
Test  positions: 4000


## Train Tuned Lenses

Three variants:
- **`tuned_full`**: affine translator minimising KL(model_full_vocab || lens) — canonical tuned lens
- **`tuned_concept`**: minimises KL(model_concept_softmax || lens) — cheaper, concept-vocab only
- **`tuned_hmm`**: minimises KL(HMM_oracle || lens) — directly targets belief-state alignment

Memory optimisation: translators are saved to disk immediately and reloaded per-layer during evaluation,
keeping peak CPU RAM near ~64 MB regardless of layer count.

In [9]:
@torch.no_grad()
def _concept_logits_from_resid(final_resid, eval_weights, batch_size=1024):
    """Concept-only logits (no softmax) from cached final-layer residuals."""
    dev = eval_weights.device
    W_c = eval_weights.W_c.to(dev)
    b_c = eval_weights.b_c.to(dev)
    ln  = eval_weights.ln_final.to(dev)
    h   = torch.from_numpy(final_resid).to(torch.float32)
    out = []
    for s in range(0, h.shape[0], batch_size):
        batch = h[s:s+batch_size].to(dev)
        out.append((ln(batch).to(torch.float32) @ W_c + b_c).cpu().numpy())
    return np.concatenate(out, axis=0)

In [ ]:
loss_curves    = {}
trained_lenses = []

if TRAIN_FULL:
    print('=== tuned_full ===')
    tr, lc = train_tuned_lens(
        activations_by_layer=train_acts, model=model,
        layers=LAYER_INDICES, target_final_resid=train_final_resid,
        n_epochs=TL_EPOCHS, lr=TL_LR, batch_size=TL_BATCH,
        optimizer_name=TL_OPTIMIZER, use_bf16=USE_BF16,
    )
    save_translators(tr, OUTPUT_DIR / 'translators_tuned_full')
    loss_curves['tuned_full'] = lc
    trained_lenses.append('tuned_full')
    del tr; torch.cuda.empty_cache()
    print('  saved.')

if TRAIN_CONCEPT:
    print('=== tuned_concept ===')
    train_concept_logits = _concept_logits_from_resid(train_final_resid, eval_weights)
    tr, lc = train_tuned_lens_concept(
        activations_by_layer=train_acts, model=model,
        concept_ids=concept_ids, layers=LAYER_INDICES,
        target_concept_values=train_concept_logits, target_is_probs=False,
        n_epochs=TL_EPOCHS, lr=TL_LR, batch_size=TL_BATCH,
        optimizer_name=TL_OPTIMIZER, use_bf16=USE_BF16,
    )
    save_translators(tr, OUTPUT_DIR / 'translators_tuned_concept')
    loss_curves['tuned_concept'] = lc
    trained_lenses.append('tuned_concept')
    del tr, train_concept_logits; torch.cuda.empty_cache()
    print('  saved.')

if TRAIN_HMM:
    print('=== tuned_hmm ===')
    if TRAIN_POS_WINDOW is not None:
        ws, we = TRAIN_POS_WINDOW
        train_obs_flat = obs_probs[:N_TRAIN, ws:we, :].reshape(-1, obs_probs.shape[-1])
    else:
        train_obs_flat = obs_probs[:N_TRAIN, :seq_len_actual, :].reshape(-1, obs_probs.shape[-1])
    tr, lc = train_tuned_lens_concept(
        activations_by_layer=train_acts, model=model,
        concept_ids=concept_ids, layers=LAYER_INDICES,
        target_concept_values=train_obs_flat, target_is_probs=True,
        n_epochs=TL_EPOCHS, lr=TL_LR, batch_size=TL_BATCH,
        optimizer_name=TL_OPTIMIZER, use_bf16=USE_BF16,
    )
    save_translators(tr, OUTPUT_DIR / 'translators_tuned_hmm')
    loss_curves['tuned_hmm'] = lc
    trained_lenses.append('tuned_hmm')
    del tr, train_obs_flat; torch.cuda.empty_cache()
    print('  saved.')

print(f'Trained lens variants: {trained_lenses}')

=== tuned_full ===


Training tuned lens:   0%|          | 0/32 [00:00<?, ?it/s]

## Evaluate

Move backbone to CPU first to free GPU memory (≈18 GB), then evaluate using only the
small `eval_weights` tensors extracted earlier.

In [ ]:
@torch.no_grad()
def _concept_probs_from_resid(final_resid, eval_weights, batch_size=1024):
    """ln_final + concept-only unembed + softmax (fp32) from cached residuals."""
    dev = eval_weights.device
    W_c = eval_weights.W_c.to(dev)
    b_c = eval_weights.b_c.to(dev)
    ln  = eval_weights.ln_final.to(dev)
    h   = torch.from_numpy(final_resid).to(torch.float32)
    out = []
    for s in range(0, h.shape[0], batch_size):
        batch  = h[s:s+batch_size].to(dev)
        normed = ln(batch).to(torch.float32)
        out.append(F.softmax(normed @ W_c + b_c, dim=-1).cpu().numpy())
    return np.concatenate(out, axis=0)


# Free backbone GPU memory before the evaluation loop
model.cpu()
gc.collect(); torch.cuda.empty_cache()
print('Backbone moved to CPU. Evaluating ...')

final_probs        = _concept_probs_from_resid(test_final_resid, eval_weights)
test_beliefs_flat  = belief_states[N_TRAIN:, :seq_len_actual].reshape(-1, belief_states.shape[-1])

all_metrics = []
for layer in tqdm(LAYER_INDICES, desc='Evaluating layers'):
    lens_probs = {}
    lens_probs['logit'] = apply_logit_lens(test_acts[layer], eval_weights, layer=layer)

    for ln in trained_lenses:
        t = load_translator(OUTPUT_DIR / f'translators_{ln}', layer)
        lens_probs[ln] = apply_tuned_lens(test_acts[layer], t, eval_weights, layer=layer)
        del t
    torch.cuda.empty_cache()

    m = compute_layer_metrics(
        layer=layer, lens_probs=lens_probs,
        final_model_probs=final_probs,
        hmm_probs=test_obs_flat,
        next_tokens=test_next_flat,
        n_sequences=N_TEST, seq_length=seq_len_actual,
    )
    all_metrics.append(m)

    row = f'  L{layer:02d}:'
    for k in ['logit'] + trained_lenses:
        if k in m.lenses:
            row += f'  {k}[KL_h={m.lenses[k]["kl_hmm"]:.4f}, KL_f={m.lenses[k]["kl_final"]:.4f}]'
    print(row)

print('Evaluation done.')

## Belief-State R² Probe

GPU OLS via `torch.linalg.pinv`: fits a linear probe from layer activations to
full Bayesian belief states. Returns per-layer R² and per-sequence R² (for CI bands).

In [ ]:
def r2_belief_probe_gpu(train_a, test_a, train_b, test_b, layer_indices, n_test_seqs):
    dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    Y_tr = torch.from_numpy(train_b).to(dev, dtype=torch.float32)
    Y_te = torch.from_numpy(test_b).to(dev, dtype=torch.float32)
    ss_tot = ((Y_te - Y_te.mean(0, keepdim=True)) ** 2).sum().item()

    seq_len   = Y_te.shape[0] // n_test_seqs
    bdim      = Y_te.shape[1]
    Y_te_seq  = Y_te.reshape(n_test_seqs, seq_len, bdim)

    r2, r2_ps = {}, {}
    for layer in layer_indices:
        X_tr = torch.from_numpy(train_a[layer]).to(dev, dtype=torch.float32)
        X_te = torch.from_numpy(test_a[layer]).to(dev, dtype=torch.float32)
        Xb_tr = torch.cat([X_tr, torch.ones(X_tr.shape[0], 1, device=dev)], 1)
        Xb_te = torch.cat([X_te, torch.ones(X_te.shape[0], 1, device=dev)], 1)
        W    = torch.linalg.pinv(Xb_tr) @ Y_tr
        pred = Xb_te @ W
        r2[layer] = float(1.0 - ((pred - Y_te)**2).sum().item() / (ss_tot + 1e-10))

        pred_s = pred.reshape(n_test_seqs, seq_len, bdim)
        ss_r_s = ((pred_s - Y_te_seq)**2).sum(dim=(1, 2))
        ss_t_s = ((Y_te_seq - Y_te_seq.mean(1, keepdim=True))**2).sum(dim=(1, 2))
        r2_ps[layer] = (1.0 - ss_r_s / (ss_t_s + 1e-10)).cpu().numpy()
        del X_tr, X_te, Xb_tr, Xb_te, W, pred, pred_s, ss_r_s, ss_t_s
        torch.cuda.empty_cache()

    del Y_tr, Y_te, Y_te_seq
    torch.cuda.empty_cache()
    return r2, r2_ps


if TRAIN_POS_WINDOW is not None:
    ws, we = TRAIN_POS_WINDOW
    train_beliefs_flat = belief_states[:N_TRAIN, ws:we].reshape(-1, belief_states.shape[-1])
else:
    train_beliefs_flat = belief_states[:N_TRAIN, :seq_len_actual].reshape(-1, belief_states.shape[-1])

print('Computing R\u00b2 belief probe ...')
r2_per_layer, r2_per_seq_per_layer = r2_belief_probe_gpu(
    train_acts, test_acts,
    train_beliefs_flat, test_beliefs_flat,
    LAYER_INDICES, n_test_seqs=N_TEST,
)

best_l = max(r2_per_layer, key=r2_per_layer.get)
print(f'Peak R\u00b2 = {r2_per_layer[best_l]:.4f} at layer {best_l}')

for layer in LAYER_INDICES:
    print(f'  L{layer:02d}: R\u00b2={r2_per_layer[layer]:.4f}')

## Plots

In [ ]:
# Training loss curves (saved to figures/ + shown inline)
for lens_name, lc in loss_curves.items():
    plotting.plot_training_loss(lc, FIGURES_DIR / f'training_loss_{lens_name}.png', label=lens_name)

# Show first available lens inline
if loss_curves:
    lname = next(iter(loss_curves))
    lc    = loss_curves[lname]
    layers_s = sorted(lc.keys())
    fig, ax = plt.subplots(figsize=(12, 4))
    norm = plt.Normalize(vmin=min(layers_s)-5, vmax=max(layers_s))
    for l in layers_s:
        ax.plot(lc[l], alpha=0.8, lw=1, color=plt.cm.Blues(norm(l)))
    ax.set_xlabel('Epoch'); ax.set_ylabel('KL loss')
    ax.set_title(f'Training Loss — {lname}'); ax.set_yscale('log')
    ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# Per-layer scalar metrics (saved to figures/)
plotting.plot_kl_hmm_by_layer(all_metrics,         FIGURES_DIR / 'kl_hmm_by_layer.png')
plotting.plot_kl_final_by_layer(all_metrics,        FIGURES_DIR / 'kl_final_by_layer.png')
plotting.plot_nll_by_layer(all_metrics,             FIGURES_DIR / 'nll_by_layer.png')
plotting.plot_top1_agreement_by_layer(all_metrics,  FIGURES_DIR / 'top1_agreement_by_layer.png')
plotting.plot_r2_belief_by_layer(r2_per_layer,      FIGURES_DIR / 'r2_belief_by_layer.png')

avail_lenses = list(all_metrics[0].lenses.keys()) if all_metrics else []
for ln in avail_lenses:
    plotting.plot_kl_hmm_by_position(
        all_metrics, LAYER_INDICES, ln,
        FIGURES_DIR / f'kl_hmm_by_position_{ln}.png')
    plotting.plot_kl_final_by_position(
        all_metrics, LAYER_INDICES, ln,
        FIGURES_DIR / f'kl_final_by_position_{ln}.png')

plotting.plot_summary(
    all_metrics, r2_per_layer,
    FIGURES_DIR / 'summary.png',
    title=f'Tuned Lens — {PROCESS_NAME} {PROCESS_PARAMS}',
)

from IPython.display import Image, display
display(Image(str(FIGURES_DIR / 'summary.png')))

In [ ]:
# R² bar chart inline
layers_s = sorted(r2_per_layer.keys())
r2_vals  = [r2_per_layer[l] for l in layers_s]
fig, ax  = plt.subplots(figsize=(10, 4))
ax.bar(layers_s, r2_vals, color='tab:purple', alpha=0.75)
ax.set_xlabel('Layer'); ax.set_ylabel('R\u00b2')
ax.set_title(f'Belief-State Probe R\u00b2 by Layer ({PROCESS_NAME})')
ax.grid(True, alpha=0.3, axis='y'); plt.tight_layout(); plt.show()

## Save Artifacts

- `metrics.json` — per-layer scalar metrics for all lens variants
- `per_position_metrics.npz` — per-position KL curves keyed `kl_<final|hmm>_vs_<lens>_layer<L>`
- `config.json` — full run configuration
- `training_losses.json` — per-epoch KL training loss per lens × layer
- `translators_<lens>/` — saved translator state dicts (written by `save_translators` above)
- `figures/` — all plots

In [ ]:
# metrics.json
metrics_summary = []
for m in all_metrics:
    entry = {'layer': m.layer, 'r2_belief_probe': r2_per_layer.get(m.layer)}
    for lens_name, vals in m.lenses.items():
        entry[f'{lens_name}_kl_final']       = vals['kl_final']
        entry[f'{lens_name}_kl_hmm']         = vals['kl_hmm']
        entry[f'{lens_name}_nll']            = vals['nll']
        entry[f'{lens_name}_top1_agreement'] = vals['top1_agreement']
    metrics_summary.append(entry)
with open(OUTPUT_DIR / 'metrics.json', 'w') as f:
    json.dump(metrics_summary, f, indent=2)

# per_position_metrics.npz
npz = {}
for m in all_metrics:
    for ln, vals in m.lenses.items():
        npz[f'kl_final_vs_{ln}_layer{m.layer}'] = vals['kl_final_by_pos']
        npz[f'kl_hmm_vs_{ln}_layer{m.layer}']   = vals['kl_hmm_by_pos']
for layer, r2_seq in r2_per_seq_per_layer.items():
    npz[f'r2_probe_per_seq_layer{layer}'] = r2_seq
np.savez(OUTPUT_DIR / 'per_position_metrics.npz', **npz)

# config.json
with open(OUTPUT_DIR / 'config.json', 'w') as f:
    json.dump({
        'model_name': MODEL_NAME,
        'process_name': PROCESS_NAME,
        'process_params': PROCESS_PARAMS,
        'vocab_tokens': VOCAB_TOKENS,
        'seq_length': SEQ_LENGTH,
        'n_sequences': N_SEQUENCES,
        'n_train_sequences': N_TRAIN,
        'train_pos_window': TRAIN_POS_WINDOW,
        'tuned_lens_epochs': TL_EPOCHS,
        'tuned_lens_lr': TL_LR,
        'tuned_lens_batch_size': TL_BATCH,
        'tuned_lens_optimizer': TL_OPTIMIZER,
        'train_tuned_full': TRAIN_FULL,
        'train_tuned_concept': TRAIN_CONCEPT,
        'train_tuned_hmm': TRAIN_HMM,
        'layer_indices': LAYER_INDICES,
        'random_seed': RANDOM_SEED,
        'forward_chunk_size': CHUNK_SIZE,
    }, f, indent=2)

# training_losses.json
with open(OUTPUT_DIR / 'training_losses.json', 'w') as f:
    json.dump({ln: {str(k): v for k, v in lc.items()}
               for ln, lc in loss_curves.items()}, f)

n_figs = len(list(FIGURES_DIR.iterdir()))
print(f'Artifacts saved to {OUTPUT_DIR}')
print(f'  metrics.json:             {len(metrics_summary)} layers')
print(f'  per_position_metrics.npz: {len(npz)} arrays')
print(f'  config.json, training_losses.json')
print(f'  figures/: {n_figs} files')